# Circuit board planning

## Problem description
A factory wish to place $m$ chips onto a rectangular board. They wish to mimise the size of this board to minimise costs. $w_i$ represents the width of chip $i$ and $h_i$ represents the height of chip $i$.

## Problem formulation

Problem formulation similar to floor planning problem.

## Required packages

In [ ]:
import random
import math
import gurobipy as gp
import itertools
import matplotlib.pyplot as plt
import networkx as nx

## Data generation

The following code generates random "machines" (rectangular shapes). The value for `number_of_machines` can be changed. The value of `maximum_area` controls the area that machines can take, also bounding the lengths of the machines. The largest machine is a square with maximum area.

In [ ]:
# The number of machines
number_of_chips = 4
maximum_area = 100
trace_spacing = 1
chip_spacing = 2

# Chip class
class Chip():
    def __init__(self, area, width, length):
        self.area = area
        self.width = width
        self.length = length

# Generating machines
random.seed(2026)
chips = {}
lengths = [i for i in range(1, int(math.sqrt(maximum_area))+1, 1)]
for i in range(number_of_chips):
    chip_width = random.choice(lengths)
    chip_length = random.choice(lengths)
    chip_area = chip_width*chip_length
    chips[len(chips)+1] = Chip(chip_area, chip_width, chip_length)

# Sample of data
for i in range(1,min(6,number_of_chips+1)):
    print(f"Chip {i}: Area={chips[i].area}, Width={chips[i].width}, Length={chips[i].length}")

# Required connections
attempts = 0
max_attempts = 20
while True:
    G = nx.Graph()
    for chip in chips:
        G.add_node(chip)
    connections_required = []
    all_possible_connections = list(itertools.combinations([chip for chip in chips], r=2))
    random.shuffle(all_possible_connections)
    for pair in all_possible_connections:
        G.add_edge(pair[0], pair[1])
        connections_required.append(pair)
        # Check everything connected
        if nx.is_connected(G):
            break
    # Check if planar (feasible)
    if nx.is_planar(G):
        break
    attempts += 1
    if max_attempts == attempts:
        raise Exception(f"Could not construct planar graph in {max_attempts} tries. Use different seed.")

# Show connections
nx.draw(G, pos=nx.planar_layout(G), with_labels=True)

# Using the planar graph as a start solution
smallest_negative = 0
plane_scaling = 100
planar_layout = nx.planar_layout(G)
for nd in planar_layout:
    for coord in planar_layout[nd]:
        if coord < smallest_negative:
            smallest_negative = coord
new_planar_layout = {}
for nd in planar_layout:
    new_planar_layout[nd] = (
        plane_scaling*(planar_layout[nd][0]+abs(smallest_negative)),
        plane_scaling*(planar_layout[nd][1]+abs(smallest_negative)),
        )

## Problem without rotation

Find the board that can fit all of the chips, without rotation, and adding all of the required traces

In [ ]:
# Initialise model
model = gp.Model()

# Add variables for width and length
size = model.addVars(2, name="size")

# Add position variables for bottom left of chip
chip_positions = {}
for i in chips:
    chip_positions[i] = model.addVars(2, name=f"chip_{i}_position") # Chip position
    chip_positions[i][0].Start = new_planar_layout[i][0]
    chip_positions[i][1].Start = new_planar_layout[i][1]

# Add 4 variables representing the corners of each trace
trace_corners = {}
for (i,j) in connections_required:
    for t in range(3):
        trace_corners[(i,j,t)] = model.addVars(2, name=f"trace_from_{i}_to{j}_corner_{t}_position") # Trace corner


# Link the width and lengths together
for i in chips:
    model.addConstr(chip_positions[i][0] + chips[i].width <= size[0])
    model.addConstr(chip_positions[i][1] + chips[i].length <= size[1])


# Ensure that the chips do not overlap at all
chip_pair_binaries = {}
for (m1, m2) in itertools.combinations(chips, r=2):
    chip_pair_binaries[(m1,m2)] = model.addVars(4, name=f"binaries_{m1}_{m2}", vtype="b")
    model.addGenConstrIndicator(chip_pair_binaries[(m1,m2)][0], True, chip_positions[m1][0] + chips[m1].width + chip_spacing <= chip_positions[m2][0]) # m1 is to left of m2
    model.addGenConstrIndicator(chip_pair_binaries[(m1,m2)][1], True, chip_positions[m2][0] + chips[m2].width + chip_spacing <= chip_positions[m1][0]) # m1 is to left of m2
    model.addGenConstrIndicator(chip_pair_binaries[(m1,m2)][2], True, chip_positions[m1][1] + chips[m1].length + chip_spacing <= chip_positions[m2][1]) # m1 is below m2
    model.addGenConstrIndicator(chip_pair_binaries[(m1,m2)][3], True, chip_positions[m2][1] + chips[m2].length + chip_spacing <= chip_positions[m1][1]) # m2 is below m1
    model.addConstr(chip_pair_binaries[(m1,m2)].sum() >= 1) # At least one constraint above is active


# Ensure that the traces are valid
chip_trace_binaries = {}
for (i,j) in connections_required:
    for t,tup in enumerate([(i,j,i,0), (i,j,j,2),(i,j)]):
        if t == 0 or t == 1:
            chip_trace_binaries[tup] = model.addVars(4, name=f"chip_trace_{tup}", vtype="b")
            model.addConstr(chip_trace_binaries[tup].sum() == 1)
            # Trace above chip
            model.addGenConstrIndicator(chip_trace_binaries[tup][0], True, chip_positions[tup[2]][1] + chips[tup[2]].length == trace_corners[(tup[0],tup[1],tup[3])][1])
            model.addGenConstrIndicator(chip_trace_binaries[tup][0], True, trace_corners[(tup[0],tup[1],1)][0] == trace_corners[(tup[0],tup[1],tup[3])][0])
            model.addGenConstrIndicator(chip_trace_binaries[tup][0], True, chip_positions[tup[2]][0] <= trace_corners[(tup[0],tup[1],tup[3])][0])
            model.addGenConstrIndicator(chip_trace_binaries[tup][0], True, chip_positions[tup[2]][0] + chips[tup[2]].width >= trace_corners[(tup[0],tup[1],tup[3])][0])
            # Trace below chip
            model.addGenConstrIndicator(chip_trace_binaries[tup][1], True, chip_positions[tup[2]][1] == trace_corners[(tup[0],tup[1],tup[3])][1])
            model.addGenConstrIndicator(chip_trace_binaries[tup][1], True, trace_corners[(tup[0],tup[1],1)][0] == trace_corners[(tup[0],tup[1],tup[3])][0])
            model.addGenConstrIndicator(chip_trace_binaries[tup][1], True, chip_positions[tup[2]][0] <= trace_corners[(tup[0],tup[1],tup[3])][0])
            model.addGenConstrIndicator(chip_trace_binaries[tup][1], True, chip_positions[tup[2]][0] + chips[tup[2]].width >= trace_corners[(tup[0],tup[1],tup[3])][0])
            # Trace to right of chip
            model.addGenConstrIndicator(chip_trace_binaries[tup][2], True, chip_positions[tup[2]][0] + chips[tup[2]].width == trace_corners[(tup[0],tup[1],tup[3])][0])
            model.addGenConstrIndicator(chip_trace_binaries[tup][2], True, trace_corners[(tup[0],tup[1],1)][1] == trace_corners[(tup[0],tup[1],tup[3])][1])
            model.addGenConstrIndicator(chip_trace_binaries[tup][2], True, chip_positions[tup[2]][1] <= trace_corners[(tup[0],tup[1],tup[3])][1])
            model.addGenConstrIndicator(chip_trace_binaries[tup][2], True, chip_positions[tup[2]][1] + chips[tup[2]].length >= trace_corners[(tup[0],tup[1],tup[3])][1])
            # Trace to left of chip
            model.addGenConstrIndicator(chip_trace_binaries[tup][3], True, chip_positions[tup[2]][0] == trace_corners[(tup[0],tup[1],tup[3])][0])
            model.addGenConstrIndicator(chip_trace_binaries[tup][3], True, trace_corners[(tup[0],tup[1],1)][1] == trace_corners[(tup[0],tup[1],tup[3])][1])
            model.addGenConstrIndicator(chip_trace_binaries[tup][3], True, chip_positions[tup[2]][1] <= trace_corners[(tup[0],tup[1],tup[3])][1])
            model.addGenConstrIndicator(chip_trace_binaries[tup][3], True, chip_positions[tup[2]][1] + chips[tup[2]].length >= trace_corners[(tup[0],tup[1],tup[3])][1])


# Ensuring every trace corner and connection is trace-width away
spacing_variable = {}
for i, pair in enumerate(itertools.combinations(trace_corners, r = 2)):
    spacing_variable[i] = model.addVar()
    t1 = pair[0]
    t2 = pair[1]
    model.addQConstr(
        spacing_variable[i]*spacing_variable[i] 
        == (trace_corners[t1][0]-trace_corners[t2][0])*(trace_corners[t1][0]-trace_corners[t2][0])
        + (trace_corners[t1][1]-trace_corners[t2][1])*(trace_corners[t1][1]-trace_corners[t2][1])
        )
    model.addConstr(spacing_variable[i] >= trace_spacing)


# Create a list of all traces
list_of_traces = []
for (i,j) in connections_required:
    for t in range(2):
        list_of_traces.append((i,j,t,t+1))


# Ensure that the traces do not overlap each other
trace_trace_overlap_binaries = {} 
for i, (trace1, trace2) in enumerate(itertools.combinations(list_of_traces, r=2)):
    trace_trace_overlap_binaries[i] = model.addVars(4, name=f"tt_overlap_binaries_{i}", vtype="b")
    t1_p1 = (trace1[0],trace1[1],trace1[2])
    t1_p2 = (trace1[0],trace1[1],trace1[3])
    t2_p1 = (trace2[0],trace2[1],trace2[2])
    t2_p2 = (trace2[0],trace2[1],trace2[3])
    if (trace1[0],trace1[1]) == (trace2[0],trace2[1]):
        continue
    model.addConstr(trace_trace_overlap_binaries[i].sum() >= 1) # At least one constraint above is active
    # t1 will be the primary segment. We make sure t2 is out of the way
    ## right
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][0], True, trace_corners[t1_p1][0] + trace_spacing <= trace_corners[t2_p1][0])
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][0], True, trace_corners[t1_p1][0] + trace_spacing <= trace_corners[t2_p2][0])
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][0], True, trace_corners[t1_p2][0] + trace_spacing <= trace_corners[t2_p1][0])
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][0], True, trace_corners[t1_p2][0] + trace_spacing <= trace_corners[t2_p2][0])
    ## left
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][1], True, trace_corners[t1_p1][0] >= trace_corners[t2_p1][0] + trace_spacing)
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][1], True, trace_corners[t1_p1][0] >= trace_corners[t2_p2][0] + trace_spacing)
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][1], True, trace_corners[t1_p2][0] >= trace_corners[t2_p1][0] + trace_spacing)
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][1], True, trace_corners[t1_p2][0] >= trace_corners[t2_p2][0] + trace_spacing)
    ## up
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][2], True, trace_corners[t1_p1][1] + trace_spacing <= trace_corners[t2_p1][1])
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][2], True, trace_corners[t1_p1][1] + trace_spacing <= trace_corners[t2_p2][1])
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][2], True, trace_corners[t1_p2][1] + trace_spacing <= trace_corners[t2_p1][1])
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][2], True, trace_corners[t1_p2][1] + trace_spacing <= trace_corners[t2_p2][1])
    ## down
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][3], True, trace_corners[t1_p1][1] >= trace_corners[t2_p1][1] + trace_spacing)
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][3], True, trace_corners[t1_p1][1] >= trace_corners[t2_p2][1] + trace_spacing)
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][3], True, trace_corners[t1_p2][1] >= trace_corners[t2_p1][1] + trace_spacing)
    model.addGenConstrIndicator(trace_trace_overlap_binaries[i][3], True, trace_corners[t1_p2][1] >= trace_corners[t2_p2][1] + trace_spacing)


# Ensure that traces do not overlap chips
chip_trace_overlap_binaries = {}                                                                
for i,(chip, trace) in enumerate(itertools.product(chips, list_of_traces)):
    chip_trace_overlap_binaries[i] = model.addVars(4, name=f"ct_overlap_binaries_{i}", vtype="b")
    t1 = (trace[0],trace[1],trace[2])
    t2 = (trace[0],trace[1],trace[3])
    if chip == trace[0] or chip == trace[1]:
        spacing = 0
    else:
        spacing = trace_spacing
    model.addGenConstrIndicator(chip_trace_overlap_binaries[i][0], True, chip_positions[chip][0] + chips[chip].width + spacing <= trace_corners[t1][0])
    model.addGenConstrIndicator(chip_trace_overlap_binaries[i][0], True, chip_positions[chip][0] + chips[chip].width + spacing <= trace_corners[t2][0])
    model.addGenConstrIndicator(chip_trace_overlap_binaries[i][1], True, chip_positions[chip][0] >= trace_corners[t1][0] + spacing)
    model.addGenConstrIndicator(chip_trace_overlap_binaries[i][1], True, chip_positions[chip][0] >= trace_corners[t2][0] + spacing)
    model.addGenConstrIndicator(chip_trace_overlap_binaries[i][2], True, chip_positions[chip][1] + chips[chip].length + spacing <= trace_corners[t1][1])
    model.addGenConstrIndicator(chip_trace_overlap_binaries[i][2], True, chip_positions[chip][1] + chips[chip].length + spacing <= trace_corners[t2][1])
    model.addGenConstrIndicator(chip_trace_overlap_binaries[i][3], True, chip_positions[chip][1] >= trace_corners[t1][1] + spacing)
    model.addGenConstrIndicator(chip_trace_overlap_binaries[i][3], True, chip_positions[chip][1] >= trace_corners[t2][1] + spacing)
    model.addConstr(chip_trace_overlap_binaries[i].sum() >= 1) # At least one constraint above is active


# Managing length of traces
connection_lengths = {}
for (i,j) in connections_required:
    connection_lengths[(i,j,0)] = model.addVar()
    connection_lengths[(i,j,1)] = model.addVar()
    model.addQConstr(
        connection_lengths[(i,j,0)]*connection_lengths[(i,j,0)] 
        == (trace_corners[(i,j,0)][0]-trace_corners[(i,j,1)][0])*(trace_corners[(i,j,0)][0]-trace_corners[(i,j,1)][0])
        + (trace_corners[(i,j,0)][1]-trace_corners[(i,j,1)][1])*(trace_corners[(i,j,0)][1]-trace_corners[(i,j,1)][1])
        )
    model.addQConstr(
        connection_lengths[(i,j,1)]*connection_lengths[(i,j,1)] 
        == (trace_corners[(i,j,1)][0]-trace_corners[(i,j,2)][0])*(trace_corners[(i,j,1)][0]-trace_corners[(i,j,2)][0])
        + (trace_corners[(i,j,1)][1]-trace_corners[(i,j,2)][1])*(trace_corners[(i,j,1)][1]-trace_corners[(i,j,2)][1])
        )
    
# Minimise objective function
trade_off = 0.5 # Closer to 1 => care more about perimeter. Closer to 0 => care more about trace length. 
term1 = gp.quicksum(connection_lengths[pair] for pair in connection_lengths)
term2 = 2*(size[0]+size[1])
model.setObjective((1-trade_off)*term1 + (trade_off)*term2)

# Optimize model
model.setParam('TimeLimit', 5*60)
model.setParam('MIPGap', 0.05)
model.optimize()

# Print optimal
print()
print(f"Circuit board is {size[0].X}cm in width and {size[1].X}cm in length")
optimal_chip_positions = {}
for i in chips:
    optimal_chip_positions[i] = (chip_positions[i][0].X,chip_positions[i][1].X)
    print(f"chip {i} has position ({optimal_chip_positions[i][0]},{optimal_chip_positions[i][1]})")
optimal_trace_corners = {}
for (i,j) in connections_required:
    for t in range(3):
        optimal_trace_corners[(i,j,t)] = (trace_corners[(i,j,t)][0].X, trace_corners[(i,j,t)][1].X)


## Plot the board layout

In [ ]:
fig, ax = plt.subplots()
for i in chips:
    ax.add_patch(
        plt.Rectangle(
            (optimal_chip_positions[i][0], optimal_chip_positions[i][1]),
            width=chips[i].width,
            height=chips[i].length,
            linewidth=1,
            edgecolor='k',
        )
    )
# Plot the connection points and corners
for corner in optimal_trace_corners:
    plt.plot(optimal_trace_corners[corner][0], optimal_trace_corners[corner][1],"ko")

# Plot the traces
for (i,j) in connections_required:
    for t in range(2):
        x = [optimal_trace_corners[(i,j,t)][0], optimal_trace_corners[(i,j,t+1)][0]]
        y = [optimal_trace_corners[(i,j,t)][1], optimal_trace_corners[(i,j,t+1)][1]]
        plt.plot(x, y, 'ro-', linewidth=4,)

maximum_dimension = max(size[0].X, size[1].X)
scale = 0.1*maximum_dimension
ax.set_xlim([0-scale, size[0].X+scale])
ax.set_ylim([0-scale, size[1].X+scale])
plt.show()